In [5]:
# ============================================================
# CELL 1 — Spark Configuration + Imports
# GlobalWatch: Bronze Ingestion — OpenAQ API
# ============================================================

# --- Spark Optimization Settings ---
# AQE: lets Spark dynamically optimize shuffle partitions at runtime
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Coalesce small partitions after shuffle — avoids 200 tiny files
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Auto-detect and handle skewed partitions (e.g. high-volume city stations)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# --- Standard Imports ---
import requests                              # HTTP calls to OpenAQ API
from datetime import datetime, timezone, timedelta   # UTC timestamps + freshness window
from pyspark.sql import functions as F       # PySpark column functions
from pyspark.sql.types import *              # Schema type definitions

# --- Database Context ---
# Fabric encodes the lakehouse path into an internal DB name
# We let Fabric tell us its own name rather than hardcoding it
# This avoids the SCHEMA_NOT_FOUND error from name duplication
DB = spark.sql("SELECT current_database()").collect()[0][0]

# --- API Key ---
# Stored securely in Fabric Environment Spark properties
# Key: spark.openaq.api.key — never hardcoded in notebook
OPENAQ_API_KEY = spark.conf.get("spark.openaq.api.key")

print(f"Config loaded ✅")
print(f"DB context: {DB}")
print(f"API Key loaded: {'✅' if OPENAQ_API_KEY else '❌ NOT FOUND — check environment'}")

StatementMeta(, 8520d5ce-cde1-46ad-ae94-f98aa9d61ab1, 9, Finished, Available, Finished, False)

Config loaded ✅
DB context: chimcobldhq2aprcdth62r3nc5q66q1dchinc9b2e9nmsuj5btjmorr2c5m7eobkcdk2ap32ds
API Key loaded: ✅


In [6]:
# ============================================================
# CELL 2 — Watermark Control Table
# Purpose: Track last loaded date per source
# Pattern: Incremental ingestion — only pull new data each run
# ============================================================

# Create watermark table if it doesn't exist
# USING DELTA: enables ACID transactions + time travel
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB}.watermark_control (
    source_name      STRING,      -- source identifier e.g. 'openaq_batch'
    last_loaded_date DATE,        -- date of last successful load
    last_loaded_ts   TIMESTAMP    -- full timestamp of last successful load
) USING DELTA
""")

# Seed initial watermark only if table is empty
count = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {DB}.watermark_control
""").collect()[0]['cnt']

if count == 0:
    # Start from Jan 2024 — historical backfill starting point
    spark.sql(f"""
    INSERT INTO {DB}.watermark_control VALUES
    ('openaq_batch', '2024-01-01', '2024-01-01T00:00:00'),
    ('waqi_batch',   '2024-01-01', '2024-01-01T00:00:00')
    """)
    print("Initial watermark seeded ✅")
else:
    print(f"Watermark table already has {count} rows — skipping seed")

# Display current watermark state
spark.sql(f"SELECT * FROM {DB}.watermark_control").show()

# --- Bind the watermark into the session ---
# This read is what makes the table load-bearing. Without it the ingest
# cell falls through to its fallback and every run reloads everything,
# which is how readings dated 2016 reached Gold.
_wm = spark.sql(f"""
    SELECT last_loaded_ts
    FROM {DB}.watermark_control
    WHERE source_name = 'openaq_batch'
""").collect()

if not _wm or _wm[0]["last_loaded_ts"] is None:
    raise RuntimeError(
        "No watermark row for 'openaq_batch'. Refusing to run: a missing "
        "watermark previously defaulted to 2000-01-01 and silently loaded "
        "the full history."
    )

last_watermark = _wm[0]["last_loaded_ts"]
print(f"Watermark in effect: {last_watermark}")
print("Watermark table ready ✅")

StatementMeta(, 8520d5ce-cde1-46ad-ae94-f98aa9d61ab1, 10, Finished, Available, Finished, False)

Watermark table already has 2 rows — skipping seed
+------------+----------------+--------------------+
| source_name|last_loaded_date|      last_loaded_ts|
+------------+----------------+--------------------+
|openaq_batch|      2026-08-09|2026-08-09 20:37:...|
|  waqi_batch|      2024-01-01| 2024-01-01 00:00:00|
+------------+----------------+--------------------+

Watermark in effect: 2026-08-09 20:37:46.809022
Watermark table ready ✅


In [7]:
# ============================================================
# CELL 3 — Rate-limit-compliant OpenAQ API functions
# Free tier: 60 req/min, 2,000 req/hr
# Strategy: 1.05s minimum gap + header-aware pause + 429 backoff
# ============================================================

import requests
import time
from datetime import datetime, timezone, timedelta

OPENAQ_BASE = "https://api.openaq.org/v3"

_last_call_ts = 0.0

def _throttle():
    """Enforce minimum 1.05s between any two API calls (~57 req/min max)."""
    global _last_call_ts
    wait = 1.05 - (time.time() - _last_call_ts)
    if wait > 0:
        time.sleep(wait)
    _last_call_ts = time.time()

def _check_rate_headers(headers):
    """Sleep until reset if remaining budget drops to 6 or fewer."""
    remaining = int(headers.get("x-ratelimit-remaining", 999))
    reset_in  = int(headers.get("x-ratelimit-reset", 60))
    if remaining <= 6:
        print(f"  [RateLimit] {remaining} requests left — sleeping {reset_in}s until reset")
        time.sleep(reset_in + 1)

def fetch_openaq_locations(limit=50, page=1):
    """
    Fetch air quality station metadata.
    Returns station ID, name, country, coordinates, and sensor list.
    The sensor list already contains the latest reading — no second call needed.
    """
    _throttle()
    url = f"{OPENAQ_BASE}/locations"
    params  = {"limit": limit, "page": page}
    headers = {"Accept": "application/json", "X-API-Key": OPENAQ_API_KEY}
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                reset_in = int(r.headers.get("x-ratelimit-reset", 60))
                wait = (2 ** attempt) * reset_in
                print(f"  [429] Attempt {attempt+1}/3 — sleeping {wait}s")
                time.sleep(wait)
                continue
            _check_rate_headers(r.headers)
            if r.status_code == 200:
                return r.json()
            print(f"  API error {r.status_code}: {r.text[:100]}")
            return None
        except Exception as e:
            print(f"  Request failed: {e}")
            return None
    return None

def fetch_locations_for_country(country_code):
    """
    Fetch up to 40 stations for a given country via iso filter.
    1 API call per country — sensor latest values come embedded in the response.
    """
    _throttle()
    url     = f"{OPENAQ_BASE}/locations"
    params  = {"limit": 40, "iso": country_code}
    headers = {"Accept": "application/json", "X-API-Key": OPENAQ_API_KEY}
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
            if r.status_code == 429:
                reset_in = int(r.headers.get("x-ratelimit-reset", 60))
                wait = (2 ** attempt) * reset_in
                print(f"  [429] {country_code} attempt {attempt+1}/3 — sleeping {wait}s")
                time.sleep(wait)
                continue
            _check_rate_headers(r.headers)
            if r.status_code == 200:
                return r.json().get("results", [])
            print(f"  {country_code}: API error {r.status_code}")
            return []
        except Exception as e:
            print(f"  {country_code}: fetch error — {e}")
            return []
    return []

# --- Connectivity Test ---
# Uses fetch_openaq_locations — counts against rate limit (1 call)
print("Testing OpenAQ API connectivity...")
test = fetch_openaq_locations(limit=3)
if test:
    total_found = test["meta"]["found"]
    print(f"API connected ✅ — {total_found} stations globally")
    print("Sample stations:")
    for loc in test["results"]:
        code    = loc.get("country", {}).get("code", "??")
        name    = loc.get("name", "N/A")
        country = loc.get("country", {}).get("name", "N/A")
        print(f"  → [{code}] {name} | {country}")
else:
    print("❌ API connection failed — check API key in environment settings")

StatementMeta(, 8520d5ce-cde1-46ad-ae94-f98aa9d61ab1, 11, Finished, Available, Finished, False)

Testing OpenAQ API connectivity...
API connected ✅ — >3 stations globally
Sample stations:
  → [GH] NMA - Nima | Ghana
  → [GH] NMT - Nima | Ghana
  → [GH] JTA - Jamestown | Ghana


In [8]:
# ============================================================
# CELL 4 — Sequential fetch (rate-limit compliant) + Bronze write
# Key changes vs original:
#   1. ThreadPoolExecutor removed — was the source of ~5,600 extra
#      API calls per run (1 per sensor across 800 stations)
#   2. process_location() uses sensor data already in the /locations
#      response — zero additional API calls
#   3. Country loop throttled via _throttle() from Cell 3
#   4. Hourly budget guard added as safety layer
# Total API calls per run: 70 (one per country) vs ~5,670 before
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime, timezone, timedelta

TARGET_COUNTRIES = [
    # Asia (25)
    "IN", "CN", "JP", "ID", "PK", "BD", "PH", "VN", "TH", "MN",
    "KZ", "UZ", "KR", "TR", "SA", "IL", "KW", "QA", "AE", "SG",
    "MY", "LK", "NP", "MM", "KH",
    # Europe (20)
    "GB", "DE", "FR", "PL", "NL", "ES", "IT", "UA", "RU", "BE",
    "CH", "SE", "NO", "CZ", "RO", "PT", "GR", "HU", "AT", "FI",
    # Americas (15)
    "US", "BR", "MX", "CA", "AR", "CO", "PE", "CL", "EC", "BO",
    "VE", "PY", "UY", "CR", "PA",
    # Africa (10)
    "ZA", "NG", "KE", "ET", "GH", "EG", "MA", "TZ", "UG", "SN",
]

schema = StructType([
    StructField("location_id",   IntegerType(),   True),
    StructField("location_name", StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("country_code",  StringType(),    True),
    StructField("country_name",  StringType(),    True),
    StructField("latitude",      DoubleType(),    True),
    StructField("longitude",     DoubleType(),    True),
    StructField("parameter",     StringType(),    True),
    StructField("value",         DoubleType(),    True),
    StructField("unit",          StringType(),    True),
    StructField("reading_ts",    TimestampType(), True),
    StructField("ingestion_ts",  TimestampType(), True),
    StructField("source_system", StringType(),    True),
    StructField("ingestion_date", DateType(),     True),
])

HOURLY_CAP   = 1800   # self-imposed ceiling — OpenAQ free limit is 2,000/hr
STALE_AFTER_DAYS = 7

api_calls    = 0
hourly_start = time.time()

def _hourly_budget_check():
    """Pause the run if the self-imposed hourly cap is reached."""
    global api_calls, hourly_start
    api_calls += 1
    elapsed = time.time() - hourly_start
    if api_calls >= HOURLY_CAP:
        wait = max(0, 3600 - elapsed) + 5
        print(f"  [Budget] Hourly cap reached ({api_calls} calls) — sleeping {wait:.0f}s")
        time.sleep(wait)
        api_calls    = 0
        hourly_start = time.time()

def process_location(loc):
    """
    Build Bronze rows from location data already in memory.
    No additional API calls — latest sensor values are embedded
    in the /locations response under sensor['latest'].
    """
    rows = []
    now  = datetime.utcnow()

    location_id   = loc.get("id")
    location_name = loc.get("name", "")
    city          = loc.get("locality") or loc.get("country", {}).get("name", "")
    country       = loc.get("country", {}) if isinstance(loc.get("country"), dict) else {}
    country_code  = country.get("code", "")
    country_name  = country.get("name", "")
    coords        = loc.get("coordinates", {}) if isinstance(loc.get("coordinates"), dict) else {}
    sensors       = loc.get("sensors", [])

    for sensor in sensors:
        param      = sensor.get("parameter", {})
        param_name = param.get("name", "") if isinstance(param, dict) else ""
        param_unit = param.get("units", "") if isinstance(param, dict) else ""

        if param_name not in ("pm25", "pm10", "no2", "co", "o3"):
            continue

        # Latest value is embedded in the sensor object — no extra API call
        latest = sensor.get("latest", {}) or {}
        value  = latest.get("value")
        ts_str = latest.get("datetime")

        if value is None or not isinstance(value, (int, float)):
            continue

        try:
            reading_ts = datetime.strptime(ts_str[:19], "%Y-%m-%dT%H:%M:%S") if ts_str else now
        except Exception:
            reading_ts = now

        rows.append({
            "location_id":   location_id,
            "location_name": location_name,
            "city":          city,
            "country_code":  country_code,
            "country_name":  country_name,
            "latitude":      float(coords.get("latitude"))  if coords.get("latitude")  else None,
            "longitude":     float(coords.get("longitude")) if coords.get("longitude") else None,
            "parameter":     param_name,
            "value":         float(value),
            "unit":          param_unit,
            "reading_ts":    reading_ts,
            "ingestion_ts":  now,
            "source_system": "openaq_v3",
            "ingestion_date": now.date(),
        })
    return rows

# ── Step 1: Fetch locations per country (sequential, throttled) ──────────────

all_locations = []
processed_countries = 0
start_time = time.time()

print(f"Fetching stations for {len(TARGET_COUNTRIES)} countries (sequential, throttled)...")
print(f"Estimated time: ~{len(TARGET_COUNTRIES) * 1.05:.0f}s minimum")
print("-" * 60)

for country_code in TARGET_COUNTRIES:
    _hourly_budget_check()
    locs = fetch_locations_for_country(country_code)   # throttle is inside this fn
    all_locations.extend(locs)
    processed_countries += 1
    if locs:
        print(f"  {country_code}: {len(locs)} stations")
    else:
        print(f"  {country_code}: 0 stations (skipped or error)")

print(f"\nCountries fetched : {processed_countries}")
print(f"Total stations    : {len(all_locations)}")
print(f"Total API calls   : {api_calls}")
print("-" * 60)

# ── Step 2: Build rows from in-memory location data (no API calls) ───────────

print("Building Bronze rows from location data (no additional API calls)...")

all_rows  = []
processed = 0
errors    = 0

for loc in all_locations:
    try:
        rows = process_location(loc)
        all_rows.extend(rows)
        processed += 1
    except Exception as e:
        errors += 1
        print(f"  process_location error: {e}")

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.0f}s")
print(f"Stations processed : {processed}")
print(f"Errors             : {errors}")
print(f"Total readings     : {len(all_rows)}")
print("-" * 60)

# ── Step 3: Write to Bronze Delta ─────────────────────────────────────────────

if not all_rows:
    batch_max_reading_ts = None
    print("❌ No data collected — nothing written")
else:
    rows_rdd = spark.sparkContext.parallelize(all_rows)
    new_df   = spark.createDataFrame(rows_rdd, schema=schema)

    if "last_watermark" not in globals():
        raise RuntimeError(
            "last_watermark is not set — run the watermark control cell first."
        )

    # Drop readings older than STALE_AFTER_DAYS — dead stations still respond
    # with years-old values; a per-sensor high-water mark would permanently
    # exclude slow-reporting but healthy stations, so we bound on absolute age.
    cutoff = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(days=STALE_AFTER_DAYS)
    fetched = new_df.count()
    new_df  = new_df.filter(F.col("reading_ts") >= F.lit(cutoff)).cache()
    new_count = new_df.count()

    print(f"Fetched            : {fetched}")
    print(f"Cutoff             : {cutoff} ({STALE_AFTER_DAYS}d)")
    print(f"Dropped as stale   : {fetched - new_count}")
    print(f"To load            : {new_count}")

    if new_count > 0:
        target = f"{DB}.raw_openaq_readings"

        if not spark.catalog.tableExists(target):
            new_df.write \
                .format("delta") \
                .partitionBy("ingestion_date") \
                .option("mergeSchema", "true") \
                .saveAsTable(target)
            print(f"✅ Created Bronze table with {new_count} rows")
        else:
            # Align source schema to target before MERGE —
            # whenNotMatchedInsertAll() expands to every target column,
            # so any column mismatch aborts the merge.
            DERIVED = {
                "year_month":   F.date_format("reading_ts", "yyyy-MM"),
                "reading_date": F.to_date("reading_ts"),
                "reading_hour": F.hour("reading_ts"),
            }

            target_schema = spark.table(target).schema
            src_df = new_df
            for field in target_schema:
                if field.name in src_df.columns:
                    continue
                expr = DERIVED.get(field.name, F.lit(None))
                src_df = src_df.withColumn(field.name, expr.cast(field.dataType))
                how = "derived" if field.name in DERIVED else "null-filled"
                print(f"  schema align: {field.name} ({how})")

            dropped = [c for c in src_df.columns if c not in [f.name for f in target_schema]]
            if dropped:
                print(f"  ⚠️ source columns not in target, dropped: {dropped}")

            src_df = src_df.select([f.name for f in target_schema])

            before = spark.table(target).count()
            DeltaTable.forName(spark, target).alias("t").merge(
                src_df.alias("s"),
                "t.location_id = s.location_id "
                "AND t.parameter  = s.parameter "
                "AND t.reading_ts = s.reading_ts"
            ).whenNotMatchedInsertAll().execute()
            after = spark.table(target).count()
            print(f"✅ Merged: {after - before} new rows, "
                  f"{new_count - (after - before)} already present")

        batch_max_reading_ts = new_df.agg(F.max("reading_ts")).collect()[0][0]
    else:
        batch_max_reading_ts = None
        print("⚠️ Nothing within the freshness window — nothing written")

StatementMeta(, 8520d5ce-cde1-46ad-ae94-f98aa9d61ab1, 12, Finished, Available, Finished, False)

Fetching stations for 70 countries...
Target countries: IN, CN, JP, ID, PK, BD, PH, VN, TH, MN, KZ, UZ, KR, TR, SA, IL, KW, QA, AE, SG, MY, LK, NP, MM, KH, GB, DE, FR, PL, NL, ES, IT, UA, RU, BE, CH, SE, NO, CZ, RO, PT, GR, HU, AT, FI, US, BR, MX, CA, AR, CO, PE, CL, EC, BO, VE, PY, UY, CR, PA, ZA, NG, KE, ET, GH, EG, MA, TZ, UG, SN
------------------------------------------------------------
  IN: 40 stations found
  CN: 40 stations found
  JP: 40 stations found
  ID: 40 stations found
  PK: 40 stations found
  BD: 22 stations found
  PH: 40 stations found
  VN: 40 stations found
  TH: 40 stations found
  MN: 40 stations found
  KZ: 40 stations found
  UZ: 5 stations found
  KR: 40 stations found
  TR: 40 stations found
  SA: 9 stations found
  IL: 40 stations found
  KW: 2 stations found
  QA: 1 stations found
  AE: 31 stations found
  SG: 40 stations found
  MY: 18 stations found
  LK: 4 stations found
  NP: 40 stations found
  MM: 3 stations found
  KH: 27 stations found
  GB: 40 s

In [9]:
# ============================================================
# CELL 5 — Validation + Watermark Update
# ============================================================

# --- Row Count ---
total = spark.sql(f"""
    SELECT COUNT(*) as total FROM {DB}.raw_openaq_readings
""").collect()[0]['total']
print(f"Total rows in Bronze: {total}")
assert total > 0, "❌ VALIDATION FAILED: Bronze table is empty!"

# --- Data Quality Summary ---
print("\nTop 10 by reading count:")
spark.sql(f"""
    SELECT
        country_code,
        country_name,
        parameter,
        ROUND(AVG(value), 2)  AS avg_value,
        ROUND(MIN(value), 2)  AS min_value,
        ROUND(MAX(value), 2)  AS max_value,
        COUNT(*)              AS readings
    FROM {DB}.raw_openaq_readings
    GROUP BY country_code, country_name, parameter
    ORDER BY readings DESC
    LIMIT 10
""").show(truncate=False)

# --- Partition Check ---
print("Partitions written:")
spark.sql(f"""
    SELECT ingestion_date, COUNT(*) as rows
    FROM {DB}.raw_openaq_readings
    GROUP BY ingestion_date
""").show()

# --- Update Watermark ---
# Derive it from the data, never from the clock. reading_ts always trails
# run time, so a current_timestamp() watermark would exclude every row on
# the following run and the pipeline would go quietly empty.
_batch_max = globals().get("batch_max_reading_ts")

if _batch_max is None:
    print("No rows loaded this run — watermark left unchanged")
else:
    spark.sql(f"""
        UPDATE {DB}.watermark_control
        SET last_loaded_ts   = TIMESTAMP'{_batch_max}',
            last_loaded_date = DATE'{_batch_max.date()}'
        WHERE source_name = 'openaq_batch'
    """)
    print(f"Watermark advanced to {_batch_max} ✅")
print("Bronze ingestion complete ✅")
print("Next: Run 04_silver_transform notebook")

StatementMeta(, 8520d5ce-cde1-46ad-ae94-f98aa9d61ab1, 13, Finished, Available, Finished, False)

Total rows in Bronze: 2521

Top 10 by reading count:
+------------+--------------+---------+---------+---------+---------+--------+
|country_code|country_name  |parameter|avg_value|min_value|max_value|readings|
+------------+--------------+---------+---------+---------+---------+--------+
|NL          |Netherlands   |         |-145.57  |-999.0   |934.0    |117     |
|CL          |Chile         |         |140.67   |0.0      |2906.65  |81      |
|US          |United States |o3       |0.03     |0.01     |0.05     |72      |
|GB          |United Kingdom|no2      |17.88    |0.0      |64.0     |65      |
|NL          |Netherlands   |pm10     |14.59    |6.46     |23.0     |60      |
|GB          |United Kingdom|pm25     |8.09     |1.0      |31.0     |59      |
|NL          |Netherlands   |no2      |8.13     |0.0      |19.2     |57      |
|GB          |United Kingdom|pm10     |14.07    |5.0      |34.0     |53      |
|NL          |Netherlands   |pm25     |7.4      |2.63     |16.6     |47      |